In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.optimize import minimize
import joblib
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import numpy as np
import random

%matplotlib inline

# Preprocessing

In [ ]:
def nfiPreprocessing(df, output_name):
    # 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
    code_species_dict = {11: ['소나무'], 12:['잣나무', '섬잣나무', '눈잣나무', '스트로브잣나무'], 13: ['일본잎갈나무', '잎갈나무'], 14: ['리기다소나무', '리기테다소나무', '방크스소나무'],
                        15: ['곰솔'], 16: ['전나무', '구상나무', '분비나무'], 17: ['편백', '화백'], 18: ['삼나무', '낙우송','메타세콰이아'], 19: ['가문비나무', '독일가문비나무', '종비나무'],
                        20: ['비자나무', '개비자나무'], 21: ['은행나무'], 31: ['상수리나무'], 32: ['신갈나무'], 33: ['굴참나무'], 34: ['갈찬나무', '떡갈나무', '졸참나무'],
                        35: ['오리나무', '물오리나무', '사방오리'], 36: ['고로쇠나무'], 37: ['자작나무', '거제수나무'],  38: ['박달나무', '개박달나무', '물박달나무'], 39: ['밤나무'],
                        40: ['물푸레나무', '들메나무', '물들메나무'], 41: ['서어나무', '개서어나무'], 42: ['때죽나무', '쪽동백나무'], 43: ['호두나무', '가래나무'], 44:['백합나무'], 
                        45: ['미루나무', '은사시나무', '이태리포플러나무', '수원사시나무'], 46: ['벚나무', '양벚나무', '산벚나무', '꽃벚나무', '왕벚나무', '잔털벚나무', '개벚나무', '올벚나무', '섬벚나무', '섬개벚나무', '산개벚지나무', '개벚지나무', ''], 47: ['느티나무'],  48:['층층나무', '곰의말채나무'],
                        49: ['아까시나무'], 61: ['가시나무', '붉가시나무', '종가시나무', '참가시나무', '개가시나무'], 62: ['구실잣밤나무'], 63: ['녹나무'], 64: ['굴거리나무'], 65: ['황칠나무'], 66: ['사스레피나무'], 67: ['후박나무'],
                         68: ['새덕이', '참식나무', '생달나무']}
    id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]
    name_lst1 = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', '포플러', '벚나무', 
                 '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
    name_lst2 = ['기타침엽수', '기타 참나무류', '기타활엽수']
    code_name_dict = {i: j for i, j in zip(id_lst, name_lst1)}

    # 임상도에 따라 NFI 수종명 재분류
    print(r"Reclassify based on Imsang code...")
    nfi_names = df['수종명'].unique()
    nfi_imsang = [df.loc[(df['수종명']==name), '침활구분'].unique()[0] for name in nfi_names]
    nfi_dict = {i : j for i, j in zip(nfi_names, nfi_imsang)}

    # 속성 추출 및 단위 환산
    print("Extract necessary columns & Convert Unit...")
    df2 = df[['표본점번호', '조사차기', '수종명', '침활구분', '흉고직경', '수고', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']]
    cm_to_inch = 0.3937
    cm_to_ft = 0.0328084
    df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
    df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
    df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)
    df2['해발고(m)'] = df2['해발고(m)'] / 100 # hm로 변환
    df2['경사(degree)'] = np.tan(np.radians(df2['경사(degree)'])) # tangent로 변환
    df2['방위각(º)'] = np.radians(df2['방위각(º)']) # radian으로 변환
    df2['평균수관밀도(%)'] = df2['평균수관밀도(%)'] / 100 # 소수점 자릿수로 변환
    # ['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']
    df2.columns = ['SampleID', 'Cycle', 'Species', 'Imsang', 'DBH(inch)', 'H(ft)', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long']
    df2.info

    # 수관높이비율(Crown ratio) 생성
    print("Add new columns: Crown Ratio, Crown Height, Imsang code, Imsang species name...")
    df2.insert(6, 'CR', (df2['CBH(ft)'] / df2['H(ft)']))
    df2.insert(7, 'CH', (df2['H(ft)'] - df2['CBH(ft)']))
    # 임상도 기준 수종명 및 수종코드 칼럼 삽입
    df2['I_Species'] = np.full(len(df2), '-99')
    df2['SID'] = np.full(len(df2), -99)
    for key, name_lst in code_species_dict.items(): 
        condition = df2['Species'].isin(name_lst)
        df2.loc[condition, 'I_Species'] = code_name_dict[key]
        df2.loc[condition, 'SID'] = key
        
        condition2 = ((df2['Imsang'] == '활엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition2, 'I_Species'] = '기타활엽수'
        df2.loc[condition2, 'SID'] = 30
    
        condition3 = ((df2['Imsang'] == '침엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition3, 'I_Species'] = '기타침엽수'
        df2.loc[condition3, 'SID'] = 10

    print(f"Save the dataframe at {output_name}")
    df_fin = pd.concat([df2[df2.columns[-2:]], df2[df2.columns[:3]], df2[df2.columns[3:15]]], axis=1)
    print(df_fin.shape)
    df_fin = df_fin.dropna()
    df_fin.to_csv(os.path.join(output_name), encoding='cp949', index=False)

    return df_fin

In [ ]:
df_clean = nfiPreprocessing(df, r"D:/ForestFire/CBH/data/NFI6-7_cleaned.csv")

In [ ]:
df_clean.info()

In [ ]:
df_clean.head(3)

# Funciton

In [ ]:
# 함수 정의
# Hansenauer & Monserud, 1996 변형
def func4(X, a1, a2, a3, b):
    H, D = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    X = (a1 * H_log/D_log)+(a2 * H_log)+(a3 * D_log**2) + b
    cr = 1 / (1 + np.exp(-X))
    return cr
# loss function for func3
def loss_func4(params, lam, X, y):
    y_pred = func4(X, *params)
    return np.sum((y - y_pred) ** 2) + lam * np.sum(params**2) # L2 규제 적용

# Model Building: Allometric

In [ ]:
# read training dataset
data_dir = r"D:\ForestFire\CBH\data"
file_name = r"NFI6-7_cleaned2.csv"
df_all = pd.read_csv(os.path.join(data_dir, file_name), encoding='cp949')
df_all = df_all.reset_index()

In [ ]:
# Map names to codes
name_lst1 =  ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', 
             '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', 
             '포플러', '벚나무', '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
name_lst2 = ['기타침엽수', '기타참나무류', '기타활엽수']  # same as your input
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]     # same as your input
code_name_dict = {j: i for i, j in zip(id_lst, name_lst1)}
code_name_dict.update({'기타침엽수': 10, '기타활엽수': 30, '기타참나무류': 34})
reversed_dict = {value : key for key, value in code_name_dict.items()}

In [ ]:
# Configuration
try_num = "HM변형-try1.1-일부"

func = func4
loss_func = loss_func4

lam = 0
test_ratio = 0.15
n_fold = 5
params_num = 4
cols = ['DBH(inch)', 'H(ft)', 'CR', 'Cycle']

result_dir = r'D:/ForestFire/CBH/result/Baseline3'
os.makedirs(result_dir, exist_ok=True)


target_trees = [16, 37, 44, 62, 68] # np.unique(df_all.SID) # [code_name_dict[i] for i in valid_species]
rec = np.zeros((len(target_trees), 17 + params_num), dtype=object)
rec[:, 0] = target_trees

# Collector for unified test set
total_test_list = []
total_train_list = []

for i, sid in enumerate(tqdm(target_trees), 1):
    print(f"[{i}/{len(target_trees)}] Processing SID: {sid}")
    condition = (df_all['SID'] == sid)
    nfi6 = df_all.query("Cycle == 6").loc[condition]
    nfi7 = df_all.query("Cycle == 7").loc[condition]

    cnt = (len(nfi6), len(nfi7))
    print(cnt)
    if nfi6.isnull().values.any() or nfi7.isnull().values.any():
        print("Null data detected, skipping SID:", sid)
        continue

    # Split train/test
    if len(nfi6) >= 180:
        nfi6_test = nfi6.sample(frac=test_ratio, random_state=SEED)
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = pd.concat([nfi6.drop(nfi6_test.index).assign(Cycle=6),nfi7.drop(nfi7_test.index).assign(Cycle=7)])

    elif (len(nfi6) < 180) & (len(nfi7) >= 30):
        nfi6_test = nfi6
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = nfi7.drop(nfi7_test.index).assign(Cycle=7)

    else:
        print(f"Skip {sid}: Not enough number of the samples")
        best_params = [np.nan] * params_num
        rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan
        ] + list(best_params)
        continue

    # X, y array 만들기
    X = df_train[cols].drop(columns=['CR'])
    y = df_train['CR']
    stratify_col = df_train['Cycle']

    # score 변수 initialization
    best_score, worst_score, best_params = -np.inf, np.inf, None
    cv_scores = []

    print("lenght of training dataset: ", len(X))
    if len(X) >= 30:
        kf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
        for train_idx, val_idx in kf.split(X, stratify_col):
            X_train = X.iloc[train_idx].drop(columns='Cycle').values.T
            y_train = y.iloc[train_idx].values
            popt, _ = curve_fit(func, X_train, y_train, maxfev=10000)
            result_reg = minimize(loss_func, x0=popt, args=(lam, X_train, y_train))
            opt_params = result_reg.x
            score = r2_score(y_train, func(X_train, *opt_params))
            cv_scores.append(score)
            if score > best_score:
                best_score = score
                best_params = opt_params
            if score < worst_score:
                worst_score = score
    else: continue

    # Evaluate on full training data
    X_full = X.drop(columns='Cycle').values.T
    y_full = y
    y_pred_full = func(X_full, *best_params)
    r2_train = r2_score(y_full, y_pred_full)
    mae_train = mean_absolute_error(y_full, y_pred_full)
    rmse_train = root_mean_squared_error(y_full, y_pred_full)

    # Evaluate on test sets
    X6_test = nfi6_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y6_test = nfi6_test['CR']
    X7_test = nfi7_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y7_test = nfi7_test['CR']
        
    # test dataset 예측 (NFI6, NFI7, NFI6+7)
    y_pred6 = func(X6_test, *best_params)
    y_pred7 = func(X7_test, *best_params)
    
    # test dataset score 산출
    r2_test_nfi6 = r2_score(y6_test, y_pred6)
    r2_test_nfi7 = r2_score(y7_test, y_pred7)
    mae_test_nfi6 = mean_absolute_error(y6_test, y_pred6)
    mae_test_nfi7 = mean_absolute_error(y7_test, y_pred7)
    rmse_test_nfi6 = root_mean_squared_error(y6_test, y_pred6)
    rmse_test_nfi7 = root_mean_squared_error(y7_test, y_pred7)
    y_all_true = np.concatenate([y6_test, y7_test])
    y_all_pred = np.concatenate([y_pred6, y_pred7])
    r2_test_all = r2_score(y_all_true, y_all_pred)
    mae_test_all = mean_absolute_error(y_all_true, y_all_pred)
    rmse_test_all = root_mean_squared_error(y_all_true, y_all_pred)
   
    # test-train dataset list에 저장
    df_train['CR_pred'] = np.clip(y_pred_full, 0, 1)
    total_train_list.extend([df_train])
    nfi6_test['CR_pred'] = np.clip(y_pred6, 0, 1)
    nfi7_test['CR_pred'] = np.clip(y_pred7, 0, 1)
    total_test_list.extend([nfi6_test, nfi7_test])

    # record list에 score 결과 저장
    rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.mean(cv_scores), best_score, worst_score,
        r2_train, mae_train, rmse_train,
        r2_test_all, mae_test_all, rmse_test_all,
        r2_test_nfi6, mae_test_nfi6, rmse_test_nfi6,
        r2_test_nfi7, mae_test_nfi7, rmse_test_nfi7
    ] + list(best_params)

# Save results into a textfile
header = ['SID', 'Count', 'CV_Mean', 'CV_Best', 'CV_Worst',
        'R2_Train', 'MAE_Train', 'RMSE_Train',
        'R2_Test_All', 'MAE_Test_All', 'RMSE_Test_All',
        'R2_Test_NFI6', 'MAE_Test_NFI6', 'RMSE_Test_NFI6',
        'R2_Test_NFI7', 'MAE_Test_NFI7', 'RMSE_Test_NFI7']
np.savetxt(
    os.path.join(result_dir, f'Evaluation_{try_num}.txt'),
    rec, delimiter=',', fmt='%s',
    header=','.join(header + [f'Coef{i+1}' for i in range(11)]),
    comments=''
)

# Save unified test dataset
if total_test_list:
    pd.concat(total_test_list).to_csv(os.path.join(result_dir, f"NFI6+7_test_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

# Save unified train dataset
if total_train_list:
    pd.concat(total_train_list).to_csv(os.path.join(result_dir, f"NFI6+7_train_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")


In [ ]:
# Global random seed (random operations: numpy, pandas, StratifiedKfold, train_test_split)
SEED = 100 # default: 100
random.seed(SEED)
np.random.seed(SEED)

# Configuration
try_num = "HM변형-try1.4-일부"

func = func4
loss_func = loss_func4

lam = 0
test_ratio = 0.3 # default: 0.15
n_fold = 5
params_num = 4
cols = ['DBH(inch)', 'H(ft)', 'CR', 'Cycle']

result_dir = r'D:/ForestFire/CBH/result/Baseline3'
os.makedirs(result_dir, exist_ok=True)


target_trees = [16] # np.unique(df_all.SID) # [code_name_dict[i] for i in valid_species]
rec = np.zeros((len(target_trees), 17 + params_num), dtype=object)
rec[:, 0] = target_trees

# Collector for unified test set
total_test_list = []
total_train_list = []

for i, sid in enumerate(tqdm(target_trees), 1):
    print(f"[{i}/{len(target_trees)}] Processing SID: {sid}")
    condition = (df_all['SID'] == sid)
    nfi6 = df_all.query("Cycle == 6").loc[condition]
    nfi7 = df_all.query("Cycle == 7").loc[condition]

    cnt = (len(nfi6), len(nfi7))
    print(cnt)
    if nfi6.isnull().values.any() or nfi7.isnull().values.any():
        print("Null data detected, skipping SID:", sid)
        continue

    # Split train/test
    if len(nfi6) >= 180:
        nfi6_test = nfi6.sample(frac=test_ratio, random_state=SEED)
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = pd.concat([nfi6.drop(nfi6_test.index).assign(Cycle=6),nfi7.drop(nfi7_test.index).assign(Cycle=7)])

    elif (len(nfi6) < 180) & (len(nfi7) >= 30):
        # nfi6_test = nfi6
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = nfi7.drop(nfi7_test.index).assign(Cycle=7)

    else:
        print(f"Skip {sid}: Not enough number of the samples")
        best_params = [np.nan] * params_num
        rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan
        ] + list(best_params)
        continue

    # X, y array 만들기
    X = df_train[cols].drop(columns=['CR'])
    y = df_train['CR']
    stratify_col = df_train['Cycle']

    # score 변수 initialization
    best_score, worst_score, best_params = -np.inf, np.inf, None
    cv_scores = []

    print("lenght of training dataset: ", len(X))
    if len(X) >= 30:
        kf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
        for train_idx, val_idx in kf.split(X, stratify_col):
            X_train = X.iloc[train_idx].drop(columns='Cycle').values.T
            y_train = y.iloc[train_idx].values
            popt, _ = curve_fit(func, X_train, y_train, maxfev=10000)
            result_reg = minimize(loss_func, x0=popt, args=(lam, X_train, y_train))
            opt_params = result_reg.x
            score = r2_score(y_train, func(X_train, *opt_params))
            cv_scores.append(score)
            if score > best_score:
                best_score = score
                best_params = opt_params
            if score < worst_score:
                worst_score = score
    else: continue

    # Evaluate on full training data
    X_full = X.drop(columns='Cycle').values.T
    y_full = y
    y_pred_full = func(X_full, *best_params)
    r2_train = r2_score(y_full, y_pred_full)
    mae_train = mean_absolute_error(y_full, y_pred_full)
    rmse_train = root_mean_squared_error(y_full, y_pred_full)

    # Evaluate on test sets
    # X6_test = nfi6_test[cols].drop(columns=['CR', 'Cycle']).values.T
    # y6_test = nfi6_test['CR']
    X7_test = nfi7_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y7_test = nfi7_test['CR']
        
    # test dataset 예측 (NFI6, NFI7, NFI6+7)
    # y_pred6 = func(X6_test, *best_params)
    y_pred7 = func(X7_test, *best_params)
    
    # test dataset score 산출
    r2_test_nfi6 = np.nan # r2_score(y6_test, y_pred6)
    r2_test_nfi7 = r2_score(y7_test, y_pred7)
    mae_test_nfi6 = np.nan # mean_absolute_error(y6_test, y_pred6)
    mae_test_nfi7 = mean_absolute_error(y7_test, y_pred7)
    rmse_test_nfi6 = np.nan # root_mean_squared_error(y6_test, y_pred6)
    rmse_test_nfi7 = root_mean_squared_error(y7_test, y_pred7)
    y_all_true = y7_test
    y_all_pred = y_pred7
    r2_test_all = r2_score(y_all_true, y_all_pred)
    mae_test_all = mean_absolute_error(y_all_true, y_all_pred)
    rmse_test_all = root_mean_squared_error(y_all_true, y_all_pred)
   
    # test-train dataset list에 저장
    df_train['CR_pred'] = np.clip(y_pred_full, 0, 1)
    total_train_list.extend([df_train])
    nfi6_test['CR_pred'] = np.clip(y_pred6, 0, 1)
    nfi7_test['CR_pred'] = np.clip(y_pred7, 0, 1)
    total_test_list.extend([nfi6_test, nfi7_test])

    # record list에 score 결과 저장
    rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.mean(cv_scores), best_score, worst_score,
        r2_train, mae_train, rmse_train,
        r2_test_all, mae_test_all, rmse_test_all,
        r2_test_nfi6, mae_test_nfi6, rmse_test_nfi6,
        r2_test_nfi7, mae_test_nfi7, rmse_test_nfi7
    ] + list(best_params)

# Save results into a textfile
header = ['SID', 'Count', 'CV_Mean', 'CV_Best', 'CV_Worst',
        'R2_Train', 'MAE_Train', 'RMSE_Train',
        'R2_Test_All', 'MAE_Test_All', 'RMSE_Test_All',
        'R2_Test_NFI6', 'MAE_Test_NFI6', 'RMSE_Test_NFI6',
        'R2_Test_NFI7', 'MAE_Test_NFI7', 'RMSE_Test_NFI7']
np.savetxt(
    os.path.join(result_dir, f'Evaluation_{try_num}.txt'),
    rec, delimiter=',', fmt='%s',
    header=','.join(header + [f'Coef{i+1}' for i in range(11)]),
    comments=''
)

# Save unified test dataset
if total_test_list:
    pd.concat(total_test_list).to_csv(os.path.join(result_dir, f"NFI6+7_test_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

# Save unified train dataset
if total_train_list:
    pd.concat(total_train_list).to_csv(os.path.join(result_dir, f"NFI6+7_train_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

In [ ]:
df_result = pd.DataFrame(rec, columns=header + [f'Par{i}' for i in range(params_num)])
s_names = [reversed_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', s_names)
r2_columns = ['CV_Mean', 'CV_Best', 'CV_Worst', 'R2_Train', 'R2_Test_All', 'R2_Test_NFI6', 'R2_Test_NFI7']
df_result.loc[:, r2_columns] = df_result.loc[:, r2_columns].clip(lower=0) # .applymap(lambda x: 0 if x < 0 else x)
df_result.to_csv(os.path.join(result_dir, f'Evaluation_{try_num}.csv'), encoding='cp949')

In [ ]:
df_result.filter(regex="R2_", axis=1)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# Set font for Korean
plt.rc('font', family='Malgun Gothic')  
plt.rcParams['axes.unicode_minus'] = False  

# Read data
title = f"Evaluation_{try_num}"
df_result = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')

# Set figure
plt.figure(figsize=(20, 6))
bar_width = 0.15
x = np.arange(len(df_result))

# Plot bars side-by-side
plt.bar(x - 2*bar_width, df_result["CV_Mean"], width=bar_width, label="CV_Mean", color="blue", alpha=0.7)
plt.bar(x - bar_width, df_result["R2_Train"], width=bar_width, label="R2_Train", color="orange", alpha=0.7)
plt.bar(x, df_result["R2_Test_All"], width=bar_width, label="R2_Test_All", color="purple", alpha=0.7)
plt.bar(x + bar_width, df_result["R2_Test_NFI6"], width=bar_width, label="R2_Test_NFI6", color="pink", alpha=0.7)
plt.bar(x + 2*bar_width, df_result["R2_Test_NFI7"], width=bar_width, label="R2_Test_NFI7", color="skyblue", alpha=0.7)

# Annotate values above bars
for i in range(len(df_result)):
    plt.text(x[i] - 2*bar_width, df_result["CV_Mean"][i] + 0.01, f'{df_result["CV_Mean"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] - bar_width, df_result["R2_Train"][i] + 0.01, f'{df_result["R2_Train"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i], df_result["R2_Test_All"][i] + 0.01, f'{df_result["R2_Test_All"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + bar_width, df_result["R2_Test_NFI6"][i] + 0.01, f'{df_result["R2_Test_NFI6"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + 2*bar_width, df_result["R2_Test_NFI7"][i] + 0.01, f'{df_result["R2_Test_NFI7"][i]:.2f}', ha='center', fontsize=6)

# X labels and layout
x_labels = [f"{df_result.loc[i, 'SName']}{df_result.loc[i, 'Count']}" for i in range(len(df_result))]
plt.xticks(ticks=x, labels=x_labels, rotation=45, ha="right")
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.legend()
plt.tight_layout()

# Save and show
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()


In [ ]:
# scatter plot
train_header = "NFI6+7_train_combined"
test_header = "NFI6+7_test_combined"
df_train_pred = pd.read_csv(os.path.join(result_dir, f"{train_header}_{try_num}.csv"), encoding="cp949")
df_test_pred = pd.read_csv(os.path.join(result_dir, f"{test_header}_{try_num}.csv"), encoding="cp949")
df_train_pred['Use'] = "train"
df_test_pred['Use'] = "test"
df_merge_pred = pd.concat([df_train_pred, df_test_pred], axis=0)
df_merge_pred.info()

# sns.lmplot(data=df_merge_pred, x="CR", y = "CR_pred", hue="Use", line_kws={"linewidth" : 3, "linestyle" : "--", 'color': 'red'}) #, palette={"pred" : })
for use, df_group in df_merge_pred.groupby("Use"):
    sns.regplot(
        data=df_group,
        x="CR",
        y="CR_pred",
        scatter=True,
        label=f"{use}",
        scatter_kws={'color': 'pink' if use == 'train' else 'lightblue', 's': 30,
                    "alpha" : 0.6 if use == 'train' else 1},
        line_kws={'color': 'blue' if use == 'train' else 'red', 'linewidth': 2, 
                  "alpha" : 0.6 if use == 'train' else 1,
                 "linestyle" : ":" if use == 'train' else "-"}
    )

plt.legend()
plt.show()

# Modeling Building: Tree-based model

In [ ]:
# import ML libraries
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import shap
import sys
from xgboost import plot_importance
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
# import optuna
from functools import partial
import pandas as pd
import joblib

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

In [ ]:
print(xgb.__version__)
print(XGBRegressor)

In [ ]:
print(sys.executable)

In [ ]:
result_dir = r"D:/ForestFire/CBH/result/Baseline3"
data_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')
df.info()

In [ ]:
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
# encoded species code
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = 'SID'  # species id
target_col = 'CR'  # assuming target column name is 'CR'

# Prepare X, y, species
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# Scale only numerical features (except encoded SID)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

# Model definitions
xgb_model = xgb.XGBRegressor(random_state=SEED)
rf_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)

# Fit models
print("Model Training...")
xgb_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train_scaled, y_train)

# Predictions
print("Model prediction...")
xgb_pred = xgb_model.predict(X_val_scaled)
rf_pred = rf_model.predict(X_val_scaled)

# Evaluation Function
def evaluate_model(y_true, y_pred):
    return {
        "R²": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }

# Overall Performance
print("Model Evaluation...")
xgb_metrics = evaluate_model(y_val, xgb_pred)
rf_metrics = evaluate_model(y_val, rf_pred)

# Print Overall Comparison
print("Overall Performance Comparison")
print(pd.DataFrame([xgb_metrics, rf_metrics], index=['XGBoost', 'Random Forest']))

# Per-Species Evaluation
unique_species = species_val.unique()
records = []

for sid in unique_species:
    idx = (species_val == sid)
    if idx.sum() > 1:  # skip species with <2 samples
        record = {"SID": sid}
        # XGBoost
        record["XGB_R²"] = r2_score(y_val[idx], xgb_pred[idx])
        record["XGB_MAE"] = mean_absolute_error(y_val[idx], xgb_pred[idx])
        record["XGB_RMSE"] = np.sqrt(mean_squared_error(y_val[idx], xgb_pred[idx]))
        # Random Forest
        record["RF_R²"] = r2_score(y_val[idx], rf_pred[idx])
        record["RF_MAE"] = np.sqrt(mean_absolute_error(y_val[idx], rf_pred[idx]))
        record["RF_RMSE"] = np.sqrt(mean_squared_error(y_val[idx], rf_pred[idx]))
        records.append(record)

df_species_comparison = pd.DataFrame(records).sort_values(by="SID").reset_index(drop=True)

In [ ]:
df_species_comparison.to_csv(os.path.join(result_dir, 'temp_acc.csv'))

In [ ]:
# Feature names
feature_names = X_train_scaled.columns.tolist()

# Get feature importances
xgb_importance = xgb_model.feature_importances_
rf_importance = rf_model.feature_importances_

# Sort features by XGBoost importance (for consistent display)
sorted_idx = np.argsort(xgb_importance)[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bar_width = 0.35
x = np.arange(len(feature_names))

bars1 = ax.bar(x - bar_width/2, xgb_importance[sorted_idx], width=bar_width, label='XGBoost')
bars2 = ax.bar(x + bar_width/2, rf_importance[sorted_idx], width=bar_width, label='Random Forest')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

# Final touches
ax.set_xticks(x)
ax.set_xticklabels(sorted_features, rotation=45, ha='right')
ax.set_ylabel("Feature Importance")
ax.set_title("Feature Importance Comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_species_comparison_test = pd.DataFrame(records_test).sort_values(by="SID").reset_index(drop=True)
# df_species_comparison.columns = pd.MultiIndex.from_product([['Train'], df_species_comparison.columns])
df_species_comparison_test.columns = pd.MultiIndex.from_product([['Test'], df_species_comparison_test.columns])
df_comparison_merged = pd.concat([df_species_comparison, df_species_comparison_test], axis=1)
df_comparison_merged

# Hyperparameter tuning with XGBoost: GridSearchCV

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV, KFold, GroupKFold
from sklearn.metrics import make_scorer, r2_score

In [ ]:
# GridSearchCV
# encoded species code
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = 'SID'  # species id
target_col = 'CR'  # assuming target column name is 'CR'

# Prepare X, y, species
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# Scale only numerical features (except encoded SID)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

# Hyerparameter tuning starts...
# 1. Define the model
print("Hyperparameter tuning starts...")
model = XGBRegressor(random_state=SEED)

# 2. Define the parameter grid
param_grid = {
    # 'n_estimators': [100, 200], # number of trees
    # 'max_depth': [7] # max depth of a tree (0 ~ ∞)
    'learning_rate': [0.01, 0.1, 0.2, 0.5], # step size srhinkage used in update to prevent overfitting (srhrinkig the feature weights)(0 ~ 1)
    # 'min_child_weight': [10, 50, 100, 200] # minimum sum of instance weight or number of instances needed in a child (0 ~ ∞)
    # 'subsample': [0.8, 1.0], # ratio of the training instances (occuring once in every boosting iteration) 
    # 'colsample_bytree': [0.8, 1.0] # subsample ratio of columns for constructing each tree (occuring once for every tree constructed) (0 ~ 1)
}

# 3. Define scoring
def species_weighted_r2(y_true, y_pred, groups):
    df = pd.DataFrame({'true': y_true, 'pred': y_pred, 'group': groups})
    scores = []
    for sid, group in df.groupby('group'):
        scores.append(r2_score(group['true'], group['pred']))
    return np.mean(scores)
    
scorer = make_scorer(r2_score)

# 4. Set up GridSearchCV
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print("GridSearch strats...")
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scorer,
    cv=cv,                  # 5-fold cross-validation
    verbose=1,             # to see progress
    n_jobs=-1              # use all CPU cores
)

# 5. Fit the grid search
grid_search.fit(X_train_scaled, y_train)

# 6. Best results
print("Best parameters:", grid_search.best_params_)
print("Best R² score:", grid_search.best_score_)

# 7. Use the best model to predict
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val_scaled)

# 8. Evaluate with seperate valiation dataset
# Evaluation Function
def evaluate_model(y_true, y_pred):
    return {
        "R²": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }

# Overall Performance
print("Model Evaluation...")
val_metrics = evaluate_model(y_val, y_pred)

# Print Overall Performance
df_performance = pd.DataFrame([val_metrics], index=['XGBoost'])

# Per-Species Performances
unique_species = species_val.unique()
records = []

for sid in unique_species:
    idx = (species_val == sid)
    if idx.sum() > 1:  # skip species with <2 samples
        record = evaluate_model(y_val[idx], y_pred[idx])
        df_performance = pd.concat([df_performance, pd.DataFrame([record], index=[sid])], axis=0)

df_performance.sort_index(inplace=True)

In [ ]:
df_performance.to_csv(os.path.join(result_dir, 'temp_acc_tuned_v1.csv'), encoding='cp949')

In [ ]:
# Feature names
feature_names = X_train_scaled.columns.tolist()

# Get feature importances
xgb_importance = best_model.feature_importances_

# Sort features by XGBoost importance (for consistent display)
sorted_idx = np.argsort(xgb_importance)[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bar_width = 0.35
x = np.arange(len(feature_names))

bars1 = ax.bar(x - bar_width/2, xgb_importance[sorted_idx], width=bar_width, label='XGBoost')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

# Final touches
ax.set_xticks(x)
ax.set_xticklabels(sorted_features, rotation=45, ha='right')
ax.set_ylabel("Feature Importance")
ax.set_title("Feature Importance Comparison")
ax.legend()
plt.tight_layout()
plt.show()

# Hyperparameter Tuning: Optuna

In [ ]:
# species imbalance
species_series = df['SID']
species_counts = species_series.value_counts().sort_values(ascending=False)

pd.Series(species_series).value_counts().sort_values(ascending=False).plot(
    kind='bar',
    figsize=(12, 6),
    title='Species Sample Count (Class Imbalance)',
    xlabel='Species',
    ylabel='Number of Samples',
    rot=45,
    grid=True,
    color='skyblue'
)

# Hugely imbalance dataset

In [ ]:
# Custom R² scorer
def species_weighted_r2(y_true, y_pred, groups):
    df = pd.DataFrame({'true': y_true, 'pred': y_pred, 'group': groups})
    group_cnt_lst = df['group'].value_counts()

    weights = 1 / group_cnt_lst
    weights /= weights.sum() # == weights / weights.sum()

    scores = []
    for sid, group_df in df.groupby('group'):
        if len(group_df) >= 2:
            r2 = r2_score(group_df['true'], group_df['pred'])
            weighted = r2 * weights[sid]
            scores.append(weighted)
    return sum(scores) if scores else -np.inf

# Optuna objective
def objective(trial, kf_split, SEED, X, y, groups):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": 500,
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_categorical("min_child_weight", [10, 50, 100, 200]),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
        # "eval_metric" : "rmse"
        # "callbacks" : [xgb.callback.EarlyStopping(rounds=30, save_best=True)]
    }

    gkf = GroupKFold(n_splits=kf_split)
    scores = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        group_val = groups.iloc[val_idx]
        
        model = XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            # early_stopping_rounds=30,
            eval_metric = "rmse",
            callbacks=[xgb.callback.EarlyStopping(rounds=30, save_best=True)],
            verbose=False
        )
        
        # Safe best_iteration logging
        if hasattr(model, "best_iteration"):
            trial.set_user_attr("best_iteration", model.best_iteration)
        else:
            trial.set_user_attr("best_iteration", None)

        y_pred = model.predict(X_val)
        score = species_weighted_r2(y_val, y_pred, group_val)
        scores.append(score)
    mean_w_r2 = np.mean(scores) 
    trial.set_user_attr("weighted_r2",mean_w_r2)

    return -mean_w_r2

# Prepare data
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = 'SID'
target_col = 'CR'

X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Run Optuna
optuna_objective = partial(objective, kf_split=5, SEED=SEED, X=X, y=y, groups=species)
n_trials = 100

study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=n_trials, show_progress_bar=True)

print("Best parameters:", -study.best_params)
print("Best score:", study.best_value)

# 지금 나오는 value는 "-" 붙여서 생각해야함!

In [ ]:
df_trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
df_trials[["number", "user_attrs_weighted_r2"]]

In [ ]:
# predict with test data
best_params = study.best_params
best_params['n_estimators'] = 500
best_params['eval_metric'] = "rmse"

# read test file
test_file = r"NFI6+7_test_combined_HM변형-try1.0.csv"
df_test = pd.read_csv(os.path.join(result_dir, test_file), encoding='cp949')

# label encoding
df_test['SID_ENC'] = le.transform(df_test['SID'])

# X, Y
X_test = df_test[feature_cols]
y_test = df_test[target_col]
groups_test = df_test['SID']

# build model with best params
model = XGBRegressor(**best_params)
model.fit(X, y)

# predict with test dataset
y_pred_test = model.predict(X_test)

total_r2 = r2_score(y_test, y_pred_test)
print("R² on test: ", total_r2)

# species별 정확도

In [ ]:
groups_test = df_test['SID']
df_pred = pd.DataFrame({'true': y_test, 'pred': y_pred_test, 'group': groups_test})
group_cnt_lst = df_group['group'].value_counts()
scores = []
rmses = []
maes = []

for sid, group_df in df_pred.groupby('group'):
    if len(group_df) >= 2:
        r2 = r2_score(group_df['true'], group_df['pred'])
        rmse = mean_squared_error(group_df['true'], group_df['pred'])
        mae = mean_absolute_error(group_df['true'], group_df['pred'])
        scores.append(r2)
        rmses.append(rmse)
        maes.append(mae)

df_score= pd.DataFrame({"SID" : df_test['SID'].unique(), "I_Species" : df_test['I_Species'].unique(), "R2" : scores, "RMSE" : rmses, "MAE" : maes})
df_score.to_csv(os.path.join(result_dir, "XGBoost_HM변형-try1.0_accuracy.csv"), encoding='cp949')

In [ ]:
# save the model
model.get_params

In [ ]:
# save model
model_dir = r"D:/ForestFire/CBH/result/Baseline3/model"
joblib.dump(model, os.path.join(model_dir, "HM변형-try1.0-XGBoostGlobal.pkl"))

In [ ]:
# feature importance
# Feature names
def printFeatureImportance(df, model):
    feature_names = df.columns.tolist()
    
    # Get feature importances
    importance = model.feature_importances_
    
    # Sort features by XGBoost importance (for consistent display)
    sorted_idx = np.argsort(importance)[::-1]
    sorted_features = [feature_names[i] for i in sorted_idx]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bar_width = 0.35
    x = np.arange(len(feature_names))
    
    bars1 = ax.bar(x - bar_width/2, importance[sorted_idx], width=bar_width, label='XGBoost')
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)
    
    # Final touches
    ax.set_xticks(x)
    ax.set_xticklabels(sorted_features, rotation=45, ha='right')
    ax.set_ylabel("Feature Importance")
    ax.set_title("Feature Importance")
    ax.legend()
    plt.tight_layout()
    plt.show()

printFeatureImportance(X_test, model)

In [ ]:
plot_importance(model, importance_type='gain')

In [ ]:
# load training dataset
result_dir = r"D:/ForestFire/CBH/result/Baseline3"
model_dir = r"D:/ForestFire/CBH/result/Baseline3/model"
data_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')

# load model
model = joblib.load(os.path.join(model_dir, "HM변형-try1.0-XGBoostGlobal.pkl"))

# label encoding
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

# split data into X and Y
feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
target_col = 'CR'  # assuming target column name is 'CR'
X = df[feature_cols]
y = df[target_col]

# calculate SHAP value
explainer = shap.Explainer(model)
shap_values = explainer(X)

In [ ]:
xg_pred = model.predict(X)
df['XG_pred'] = xg_pred

In [ ]:
# global view
plt.figure(figsize=(20, 10))
shap.summary_plot(shap_values, X)

In [ ]:
df['XG_pred'] = y

# Save global result & data augmentation

In [ ]:
df['XG_residual'] = df['CR'] - df['XG_pred']
df.sort_values(by='XG_residual')

In [ ]:
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['']

In [ ]:
# Set Korean font (you can adjust to match your system's available fonts)
plt.rcParams['font.family'] = 'Malgun Gothic'  # For Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # For macOS
plt.rcParams['axes.unicode_minus'] = False

# Unique species list
species = df['I_Species'].unique()
n_species = len(species)

# Grid setup
nrow = 5
ncol = int(np.ceil(n_species / nrow))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3, nrow * 2.5))
axes = axes.flatten()

# Plotting
for i, sp in enumerate(species):
    df_s = df[df['I_Species'] == sp]
    
    # Base axis for CR (upward)
    ax = axes[i]
    ax.hist(df_s['CR'], alpha=0.6, color='skyblue', label='CR')
    ax.hist(df_s['CR_pred'], alpha=0.6, color='purple', label='Al_pred')
    ax.hist(df_s['XG_pred'], alpha=0.6, color='salmon', label='XG_pred')
    ax.set_title(sp)
    
    # Twin axis for XG_pred (downward)
    """
    ax2 = ax.twinx()
    ax2.hist(df_s['XG_pred'], alpha=0.6, color='salmon', label='XG_pred')
    ax2.invert_yaxis()  # Flip the histogram down
    """
    
    axes[i].set_title(sp)
    axes[i].legend()

# Remove empty subplots if any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Grid setup
nrow = 5
ncol = int(np.ceil(n_species / nrow))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3, nrow * 2.5))
axes = axes.flatten()

for i, sp in enumerate(species):
    df_s = df[df['I_Species'] == sp]
    
    # Base axis for CR (upward)
    ax = axes[i]
    ax.hist(df_s['XG_residual'], alpha=0.6, color='skyblue', label='residual')
    ax.hist(df_s['XG_pred'], alpha=0.6, color='salmon', label='XG_pred')
    ax.set_title(sp)
    ax.legend()

In [ ]:
# QQplot
from scipy.stats import probplot

nrow = 5
ncol = int(np.ceil(n_species / nrow))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3, nrow * 2.5))
axes = axes.flatten()

for i, sp in enumerate(species):
    df_s = df[df['I_Species'] == sp]
    residuals = df_s['XG_residual'].dropna()
    
    ax = axes[i]
    probplot(residuals, dist="norm", plot=ax)
    
    ax.set_title(sp)
    ax.get_lines()[1].set_color('red')
    plt.tight_layout()
    ax.legend()

In [ ]:
# scatterplot
nrow = 5
ncol = int(np.ceil(n_species / nrow))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3, nrow * 2.5))
axes = axes.flatten()

for i, sp in enumerate(species):
    ax = axes[i]
    ax.scatter(x=df_s['CR'], y=df_s['XG_pred'], color='skyblue')
    
    ax.set_title(sp)
    plt.tight_layout()
    ax.legend()

# fine-tuning by species

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Sample mock data structure for NFI (for demonstration)
np.random.seed(42)
n_samples = 100
df = pd.DataFrame({
    'Species': np.random.choice(['Pine', 'Oak', 'Maple'], size=n_samples),
    'DBH': np.random.normal(30, 10, size=n_samples),
    'Height': np.random.normal(15, 5, size=n_samples),
    'CR': np.random.uniform(0.3, 0.7, size=n_samples),
    'Latitude': np.random.uniform(35.0, 37.0, size=n_samples),
    'Longitude': np.random.uniform(126.0, 128.0, size=n_samples),
    'Elevation': np.random.normal(300, 100, size=n_samples),
    'Slope': np.random.uniform(0, 30, size=n_samples),
})

# 1. Define function to fit per-species allometric model: Height ~ DBH
def fit_species_allometric(df):
    species_models = {}
    for species, group in df.groupby('Species'):
        model = LinearRegression()
        model.fit(group[['DBH']], group['Height'])
        r2 = r2_score(group['Height'], model.predict(group[['DBH']])
        species_models[species] = {
            'model': model,
            'residual_std': residuals.std()
        }
    return species_models

# 2. Augmentation function using allometric + trait-consistent noise
def augment_data(df, models, n_aug_per_sample=3):
    augmented_rows = []
    for idx, row in df.iterrows():
        species = row['Species']
        model_info = models[species]
        base_dbh = row['DBH']

        for _ in range(n_aug_per_sample):
            # Add Gaussian noise to DBH
            aug_dbh = base_dbh + np.random.normal(0, 2)
            # Predict height using allometric model + noise
            pred_height = model_info['model'].predict([[aug_dbh]])[0]
            aug_height = pred_height + np.random.normal(0, model_info['residual_std'])

            # Perturb other features (latitude, longitude, etc.)
            aug_lat = row['Latitude'] + np.random.normal(0, 0.01)
            aug_lon = row['Longitude'] + np.random.normal(0, 0.01)
            aug_elev = row['Elevation'] + np.random.normal(0, 20)
            aug_slope = row['Slope'] + np.random.normal(0, 5)
            aug_cr = row['CR'] + np.random.normal(0, 0.05)

            augmented_rows.append({
                'Species': species,
                'DBH': aug_dbh,
                'Height': aug_height,
                'CR': aug_cr,
                'Latitude': aug_lat,
                'Longitude': aug_lon,
                'Elevation': aug_elev,
                'Slope': aug_slope
            })
    return pd.DataFrame(augmented_rows)

# Apply functions
models = fit_species_allometric(df)
augmented_df = augment_data(df, models, n_aug_per_sample=2)


# Refit the model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
# load training dataset
result_dir = r"D:/ForestFire/CBH/result/Baseline3"
model_dir = r"D:/ForestFire/CBH/result/Baseline3/model"
data_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')

# load model
model = joblib.load(os.path.join(model_dir, "HM변형-try1.0-XGBoostGlobal.pkl"))

In [ ]:
class LabelEncoderWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}

    def fit(self, X, y=None):
        for col in X.columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.encoders[col] = le
        return self

    def transform(self, X):
        X_encoded = X.copy()
        for col in X.columns:
            X_encoded[col] = self.encoders[col].transform(X[col])
        return X_encoded

    def inverse_transform(self, X):
        X_decoded = X.copy()
        for col in X.columns:
            X_decoded[col] = self.encoders[col].inverse_transform(X[col])
        return X_decoded

In [ ]:
# columns
# feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = ['SID']  # species id
target_col = 'CR'  # assuming target column name is 'CR'
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']
feature_cols = cat_col + numeric_cols

# Prepare X, y, species
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# encoded species code
le = LabelEncoderWrapper()
scaler = StandardScaler()
preprocessor = ColumnTransformer(
    transformers=[
         ('num', scaler, numeric_cols),
        ('cat', le, cat_col)
    ]
)

preprocessor.fit(X_train)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', model)
])

joblib.dump(pipeline, os.path.join(model_dir, 'HM변형1.0-XGBoostGlobal-pipeline.pkl'))

In [ ]:
# encoded species code
le = LabelEncoderWrapper()
df['SID_ENC'] = le.fit_transform(df[['SID']])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = ['SID']  # species id
target_col = 'CR'  # assuming target column name is 'CR'

# Prepare X, y, species
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# Scale only numerical features (except encoded SID)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
